In [ ]:
!pip install -U -q git+https://github.com/roboflow/trackers.git@develop
!pip install filterpy
!pip install -q gdown

In [ ]:
!mkdir -p MOT17_yolox_dets
file_id = "1BuXtPWf8QbPU_y1i2xY2IbTE-rj3l6qT"
output_name = f"yolox_{tracker_name}_detections_MOT17.zip"
!cd MOT17_yolox_dets && gdown $file_id -O $output_name

In [ ]:
zip_path = "MOT17_yolox_dets/" + output_name

!unzip {zip_path} -d MOT17_yolox_dets

!rm {zip_path}

In [1]:
!mkdir -p TrackEval/data/gt 
file_id = "1CpEMfN6BvYQh3msftSIdXEKy6vC--uQR"
output_name = "MOT17.zip"
!cd TrackEval/data/gt && gdown $file_id -O $output_name
!unzip TrackEval/data/gt/MOT17.zip -d TrackEval/data/gt
!rm TrackEval/data/gt/MOT17.zip

Downloading...
From: https://drive.google.com/uc?id=1CpEMfN6BvYQh3msftSIdXEKy6vC--uQR
To: /Users/alexanderbodner/Documents/roboflow/trackers metrics/mot17/TrackEval/data/gt/MOT17.zip
100%|██████████████████████████████████████| 1.53M/1.53M [00:00<00:00, 8.39MB/s]
Archive:  TrackEval/data/gt/MOT17.zip
   creating: TrackEval/data/gt/MOT17/
  inflating: TrackEval/data/gt/MOT17/MOT17-train.txt  
  inflating: TrackEval/data/gt/MOT17/MOT17-val.txt  
   creating: TrackEval/data/gt/MOT17/train_val/
   creating: TrackEval/data/gt/MOT17/train_val/MOT17-02-FRCNN/
   creating: TrackEval/data/gt/MOT17/train_val/MOT17-02-FRCNN/gt/
  inflating: TrackEval/data/gt/MOT17/train_val/MOT17-02-FRCNN/gt/gt.txt  
  inflating: TrackEval/data/gt/MOT17/train_val/MOT17-02-FRCNN/gt/gt_val_half.txt  
  inflating: TrackEval/data/gt/MOT17/train_val/MOT17-02-FRCNN/seqinfo.ini  
   creating: TrackEval/data/gt/MOT17/train_val/MOT17-04-FRCNN/
   creating: TrackEval/data/gt/MOT17/train_val/MOT17-04-FRCNN/gt/
  inflating: 

## Valid set

In [ ]:
"""
Create MOT17 GT aligned to YOLOX val detections.

For each MOT17-XX-FRCNN in `TrackEval/data/gt/MOT17/train_val`, this script:
- Reads `MOT17_yolox_dets/val/MOT17-XX_val.txt` to get the frame range where
  YOLOX val detections exist.
- Filters `gt/gt.txt` to those frames only.
- Writes the result to `TrackEval/data/gt/MOT17_yolox_val/train_val/SEQ/gt/gt.txt`.

The original MOT17 GT under `TrackEval/data/gt/MOT17` is left untouched.
"""

from __future__ import annotations

from pathlib import Path
def main() -> None:
    base_dir = Path(__file__).resolve().parent

    val_det_root = base_dir / "MOT17_yolox_dets" / "val"
    src_gt_root = base_dir / "TrackEval" / "data" / "gt" / "MOT17" / "train_val"
    dst_gt_root = base_dir / "TrackEval" / "data" / "gt" / "MOT17_yolox_val" / "train_val"

    if not src_gt_root.exists():
        raise SystemExit(f"Source GT root does not exist: {src_gt_root}")
    if not val_det_root.exists():
        raise SystemExit(f"Val detections root does not exist: {val_det_root}")

    dst_gt_root.mkdir(parents=True, exist_ok=True)

    print("Source GT root:", src_gt_root)
    print("Val det root:", val_det_root)
    print("Dest   GT root:", dst_gt_root)

    seq_dirs = sorted(p for p in src_gt_root.iterdir() if p.is_dir())
    if not seq_dirs:
        raise SystemExit(f"No sequence directories found under {src_gt_root}")

    for seq_dir in seq_dirs:
        seq_name = seq_dir.name  # e.g. MOT17-02-FRCNN
        prefix = seq_name.split("-FRCNN")[0]  # e.g. MOT17-02
        det_file = val_det_root / f"{prefix}_val.txt"

        if not det_file.exists():
            print(f"[SKIP] {seq_name}: missing det file {det_file}")
            continue

        frames: list[int] = []
        with det_file.open() as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    frame = int(line.split(",")[0])
                except ValueError:
                    continue
                frames.append(frame)

        if not frames:
            print(f"[SKIP] {seq_name}: no frames found in {det_file}")
            continue

        f_min, f_max = min(frames), max(frames)
        print(f"[INFO] {seq_name}: using frames [{f_min}, {f_max}] from {det_file.name}")

        src_gt = seq_dir / "gt" / "gt.txt"
        if not src_gt.exists():
            print(f"[SKIP] {seq_name}: missing {src_gt}")
            continue

        lines = src_gt.read_text().splitlines()
        kept: list[str] = []
        for ln in lines:
            ln = ln.strip()
            if not ln:
                continue
            try:
                frame = int(ln.split(",")[0])
            except ValueError:
                continue
            if f_min <= frame <= f_max:
                kept.append(ln)

        dst_seq_dir = dst_gt_root / seq_name
        dst_gt_dir = dst_seq_dir / "gt"
        dst_seq_dir.mkdir(parents=True, exist_ok=True)
        dst_gt_dir.mkdir(parents=True, exist_ok=True)

        # Copy non-GT files (e.g., seqinfo.ini) into dest sequence dir
        for item in seq_dir.iterdir():
            if item.name == "gt":
                continue
            if item.is_file():
                (dst_seq_dir / item.name).write_bytes(item.read_bytes())

        dst_gt_txt = dst_gt_dir / "gt.txt"
        dst_gt_txt.write_text("\n".join(kept) + ("\n" if kept else ""))

        print(f"[OK] {seq_name}: wrote {len(kept)} GT lines to {dst_gt_txt}")

    print("Done. You can now point MOT17_GT_ROOT to:")
    print("  TrackEval/data/gt/MOT17_yolox_val")


main()

Source train_val root: /Users/alexanderbodner/Documents/roboflow/trackers metrics/mot17/TrackEval/data/gt/MOT17/train_val
Dest   train_val root: /Users/alexanderbodner/Documents/roboflow/trackers metrics/mot17/TrackEval/data/gt/MOT17_val_half/train_val
[OK]  MOT17-02-FRCNN: wrote val-half GT to /Users/alexanderbodner/Documents/roboflow/trackers metrics/mot17/TrackEval/data/gt/MOT17_val_half/train_val/MOT17-02-FRCNN/gt/gt.txt
[OK]  MOT17-04-FRCNN: wrote val-half GT to /Users/alexanderbodner/Documents/roboflow/trackers metrics/mot17/TrackEval/data/gt/MOT17_val_half/train_val/MOT17-04-FRCNN/gt/gt.txt
[OK]  MOT17-05-FRCNN: wrote val-half GT to /Users/alexanderbodner/Documents/roboflow/trackers metrics/mot17/TrackEval/data/gt/MOT17_val_half/train_val/MOT17-05-FRCNN/gt/gt.txt
[OK]  MOT17-09-FRCNN: wrote val-half GT to /Users/alexanderbodner/Documents/roboflow/trackers metrics/mot17/TrackEval/data/gt/MOT17_val_half/train_val/MOT17-09-FRCNN/gt/gt.txt
[OK]  MOT17-10-FRCNN: wrote val-half GT to 